In [ ]:
# Required interpreter: Python 3.13.x
# In VS Code, create/select a Python 3.13 environment before running this notebook.
# This check runs before importing analysis dependencies.
import importlib
import modules.config as config

config = importlib.reload(config)
config.verify_requirements(install_missing=True)

from pathlib import Path
import pandas as pd

# Project configuration
from modules.config import (
    ROOT,
    PROJECT_ROOT,
    MODULES_ROOT,
    OUTPUTS_ROOT,
    IMG_ROOT,
    TAB_ROOT,
    CACHE_ROOT,
    COST_MODEL_ROOT,
    COST_SOURCES,
    EXCHANGE_ROOT,
    SUPPLY_CHAIN_ROOT,
    SUPPLY_SOURCES,
    TURBINE_DATA_ROOT,
    WINDFARM_DATA_ROOT,
    MRIO_ROOT,
    DATASETS_DIR,
    HELPERS_DIR,
    METADATA_DIR,
    MRIO_SCENARIOS_DIR,
    RESULTS_ROOT,
    CLOSE_FIGURES,
    SAVE_FIGURES,
    CORE_DEPLOYMENT,
)

PAPER_AUTHORS = "Raul Labarthe, Ulrich Elmer Hansen, and Anastasia Ioannou"
PAPER_TITLE = "Targeting Made-in-Europe rules in wind energy auctions: Which components should be covered?"
PAPER_JOURNAL = "Energy Policy"

In [ ]:
# ============================================================================
# GLOBAL CONFIGURATION FLAGS
# ============================================================================

CLOSE_FIGURES = True
SAVE_FIGURES = True
RECOMPUTE = True
FIRST_MRIO = True

# ============================================================================
# CONSTANTS
# ============================================================================

CORE_DEPLOYMENT = ["GB", "BE", "NL", "DK", "DE"]


In [ ]:
# Verify imports are loaded and show configuration summary
print("=" * 70)
print("CONFIGURATION LOADED FROM modules.config")
print("=" * 70)

print("\nROOT DIRECTORIES:")
print(f"   ROOT: {ROOT}")
print(f"   PROJECT_ROOT: {PROJECT_ROOT}")
print(f"   MODULES_ROOT: {MODULES_ROOT}")

print("\nOUTPUT DIRECTORIES:")
print(f"   OUTPUTS_ROOT: {OUTPUTS_ROOT}")
print(f"   IMG_ROOT: {IMG_ROOT}")
print(f"   TAB_ROOT: {TAB_ROOT}")
print(f"   CACHE_ROOT: {CACHE_ROOT}")
print(f"   RESULTS_ROOT: {RESULTS_ROOT}")

print("\nCOST MODEL PATHS:")
print(f"   COST_MODEL_ROOT: {COST_MODEL_ROOT}")
print(f"   COST_SOURCES: {COST_SOURCES}")
print(f"   EXCHANGE_ROOT: {EXCHANGE_ROOT}")

print("\nSUPPLY CHAIN PATHS:")
print(f"   SUPPLY_CHAIN_ROOT: {SUPPLY_CHAIN_ROOT}")
print(f"   SUPPLY_SOURCES: {SUPPLY_SOURCES}")
print(f"   TURBINE_DATA_ROOT: {TURBINE_DATA_ROOT}")
print(f"   WINDFARM_DATA_ROOT: {WINDFARM_DATA_ROOT}")

print("\nMRIO ANALYSIS PATHS:")
print(f"   MRIO_ROOT: {MRIO_ROOT}")
print(f"   DATASETS_DIR: {DATASETS_DIR}")
print(f"   HELPERS_DIR: {HELPERS_DIR}")
print(f"   METADATA_DIR: {METADATA_DIR}")
print(f"   MRIO_SCENARIOS_DIR: {MRIO_SCENARIOS_DIR}")

print("\nANALYSIS PARAMETERS:")
print(f"   - Save figures: {SAVE_FIGURES}")
print(f"   - Close figures: {CLOSE_FIGURES}")
print(f"   - Core deployment: {CORE_DEPLOYMENT}")
print("=" * 70)


### Cost Model

In [ ]:
# ============================================================================
# Cost Model Module 
# ============================================================================

import importlib
import sys
from dataclasses import dataclass

# Reload modules if already loaded
for module_name in ['modules.cost_model', 'modules.cost_model.inputs',
                    'modules.cost_model.model', 
                    'modules.cost_model.charts']:
    if module_name in sys.modules:
        try:
            importlib.reload(sys.modules[module_name])
        except:
            pass


from modules.cost_model.inputs import (
    build_fx_dictionary,
    PlantConfig,
)

from modules.cost_model.model import (
    turbine_masses,
    build_turbine_supply,
    build_cost_breakdown,
)

from modules.cost_model.charts import (
    plot_turbine_double_ring,
    plot_model_double_donut,
)

print("✓ All cost_model imported successfully")


Exchange rates


In [ ]:
# Build FX dictionary from exchange rates
# Returns empty dict if no data available (optional component)
fx_dict = build_fx_dictionary(EXCHANGE_ROOT)
print(f"✓ FX dictionary loaded with {len(fx_dict)} currencies" if fx_dict else "⚠ No FX data available (optional)")

### Turbines


Exchange rates


In [ ]:
# Exchange rates
eur_gbp = 1/fx_dict['GBP'][2024]
eur_usd= 1/fx_dict['USD'][2024] 


Turbine design


In [ ]:
@dataclass
class TurbineDesign:
    """
    European Offshore Wind Farm turbine design.

    Default values based on BVG (2025) and DTU 10MW reference turbine.
    
    """
    n_turbines: int = round(1000 / 10)  # 100 turbines for 1 GW farm
    rated_rpm: float = 9.6
    rotor_diameter: float = 178.3
    rated_power: float = 10  # MW
    hub_height: float = 119  # m
    drivetrain_type: str = "dd"  # 'dd' (direct drive), 'hs' (high-speed), 'ms' (medium-speed)
    foundation_type: str = "monopile"


Turbine cost

In [ ]:
turbine_supply_cost = build_turbine_supply(
    TurbineDesign(),
    turbine_masses(TurbineDesign()),
    eur_gbp=eur_gbp,          
    calibrate_to_bvg=True,
)


Lifetime cost breakdown


In [ ]:
# Analysis parameters
lifetime_years = 25

cost_breakdown = build_cost_breakdown(
        turbine_supply_cost,
        PlantConfig(),
        TurbineDesign(),
        project_duration_years=lifetime_years,
)

Plots

In [ ]:
# A.2 and A.3
plot_model_double_donut(
    cost_breakdown,
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
)

plot_turbine_double_ring(
    turbine_supply_cost,
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
)

In [ ]:
# Run the cost model for each country
if __name__ == "__main__":

    # Dictionary that will store results for all countries
    cost_by_country = {}

    # ---- Loop through all countries ----
    for country_deployment in CORE_DEPLOYMENT:
        print(f"▶ Running model for country: {country_deployment}")
        # Build the cost breakdown for this country
        cost_breakdown_n = build_cost_breakdown(
            turbine_supply_cost,
            PlantConfig(),
            TurbineDesign(),
            project_duration_years=lifetime_years,
        )
        cost_breakdown_n.pop("ABEX", None)
        # Store the results
        cost_by_country[country_deployment] = cost_breakdown_n

### Supply chain analysis


Turbine components procurement - WindEurope


In [ ]:
# ============================================================================
# Supply Chain Module 
# ============================================================================

import importlib
import sys

for module_name in ['modules.supply_chain.inputs',
                    'modules.supply_chain.helpers',
                    'modules.supply_chain.turbine_analysis', 
                    'modules.supply_chain.suppliers_analysis',
                    'modules.supply_chain.charts']:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

from modules.supply_chain.inputs import (
    connected_country,
    CG_STD_MAP,
)

from modules.supply_chain.turbine_analysis import (
    split_turbine_costs_by_oem,
    flatten_turbine_costs,
    add_category_to_df_turbine,
    allocate_all,
    build_turbine_trade,
    prepare_oem_share_by_country,
    oem_suppliers,
    prepare_bom_facilities,
    filter_bom_facilities_by_component_deploy,
    load_b2_turbine_inputs,
)

from modules.supply_chain.charts import (
    plot_alloc_share_stacked,
    plot_gb_local_benchmark,
    plot_macro_vertical,
)

from modules.supply_chain.inputs import bridge, ownership_oem, oem_hq_country
from modules.country import COUNTRY_COORDS

from modules.supply_chain.suppliers_analysis import (
    build_trade_matrices_from_b1_sourcing,
    build_component_flows,
    build_component_flows_checked,
)

from openpyxl import load_workbook

print("✓ All supply_chain modules imported and data loaded successfully")


In [ ]:
# ============================================================================
# Load Supply Chain Data
# ============================================================================

# Public B2 entry point: input sheets from the supplementary workbook.
# The allocation process below is still executed from these inputs.
B2_SUPPLEMENTARY_WORKBOOK = SUPPLY_SOURCES / "turbine" / "Supplementary_Material_B2_Turbine_Supply_Chain_Sourcing_Inputs.xlsx"
if not B2_SUPPLEMENTARY_WORKBOOK.exists():
    raise FileNotFoundError(
        f"Missing public B2 supplementary workbook: {B2_SUPPLEMENTARY_WORKBOOK}"
    )

b2_inputs = load_b2_turbine_inputs(B2_SUPPLEMENTARY_WORKBOOK)
turbine_orders_raw = b2_inputs["oem_orders"]
share_dict, capacity_by_country, oem_share_df, turbine_orders = prepare_oem_share_by_country(turbine_orders_raw)

# Build allocations manually
cost_by_country_oem = split_turbine_costs_by_oem(cost_by_country, share_dict)
df_turbine = pd.DataFrame(flatten_turbine_costs(cost_by_country_oem))
df_turbine["component_group"] = df_turbine["component_group"].replace(CG_STD_MAP)
df_turbine_cat = add_category_to_df_turbine(df_turbine, bridge)

In [ ]:
# OEM-specific known suppliers from B2
oem_supply_raw = b2_inputs["known_suppliers"]
summary_table, oem_supply = oem_suppliers(oem_supply_raw, component_group_map=CG_STD_MAP)

# Assembly facilities from B2
assembly_fac = b2_inputs["assembly_facilities"]

# General manufacturing facilities from B2
bom_fac_raw = b2_inputs["general_facilities"]

KEEP = {"Castings", "Forgings", "Converters", "Tower"}

bom_keep = prepare_bom_facilities(
    bom_fac_raw,
    keep=KEEP,
    include_eu27=True,
    deploy=set(CORE_DEPLOYMENT),
)

bom_facilities = filter_bom_facilities_by_component_deploy(
    bom_keep,
    df_turbine_cat,
    share_thresh=0.1,
)


Cost allocation


In [ ]:
df_alloc = allocate_all(
    df_turbine_cat=df_turbine_cat,
    oem_supply=oem_supply,
    assembly_facilities=assembly_fac,
    bom_facilities=bom_facilities,
    ownership_oem=ownership_oem,
    oem_hq_country=oem_hq_country,
    country_coords=COUNTRY_COORDS,
)

turbine_trade = build_turbine_trade(df_alloc)


In [ ]:
plot_alloc_share_stacked(df_alloc, 
                         deployment_country="BE", 
                         filename="B.2.2.pdf",
                         save=SAVE_FIGURES, 
                         close=CLOSE_FIGURES)

Other Wind Farm components - 4C Offshore


In [ ]:
# Public B1 entry point: processed sourcing_long from the supplementary workbook.
# The raw 4C Offshore pipeline is retained in modules.supply_chain.suppliers_analysis
# for licensed/internal regeneration, but it is not required for the public run.
B1_SUPPLEMENTARY_WORKBOOK = WINDFARM_DATA_ROOT / "Supplementary_Material_B1_Suppliers_Processing_Wind_Farm_4COffshore.xlsx"
CORE_ROW_ORDER = ["GB", "BE", "NL", "DK", "DE"]

if not B1_SUPPLEMENTARY_WORKBOOK.exists():
    raise FileNotFoundError(
        f"Missing public B1 supplementary workbook: {B1_SUPPLEMENTARY_WORKBOOK}"
    )

print(f"Using public B1 sourcing workbook: {B1_SUPPLEMENTARY_WORKBOOK}")


In [ ]:
# =====================================================================
# Final: flows + plots (clean pipeline)
# =====================================================================

# 1) Reviewed contract shares from public B1 sourcing_long (before local-content correction)
contract_shares_2000_2025 = build_trade_matrices_from_b1_sourcing(
    B1_SUPPLEMENTARY_WORKBOOK,
    df_alloc=df_alloc,
    period="2000-2025",
    share_version="total",
    core_row_order=CORE_ROW_ORDER,
)

# 2) Build flows BEFORE adjustment (for benchmark plot)
flows_raw, *_ = build_component_flows(
    cost_by_country=cost_by_country,
    trade_matrices=contract_shares_2000_2025,
    df_alloc=df_alloc,
)


In [ ]:
# 3) Final public sourcing shares from B1 sourcing_long.
# Uses share_total plus final_local_c to reconstruct normalized post-correction shares.
contract_shares_2000_2025_adj = build_trade_matrices_from_b1_sourcing(
    B1_SUPPLEMENTARY_WORKBOOK,
    df_alloc=df_alloc,
    period="2000-2025",
    share_version="final",
    core_row_order=CORE_ROW_ORDER,
)

# 4) Build flows AFTER adjustment + audit
flows_by_component_hier_all, flows_turbine_detail_hier, flows_turbine_L1, all_origins, missing_cols_df, colsum_issues_df = (
    build_component_flows_checked(
        cost_by_country=cost_by_country,
        trade_matrices=contract_shares_2000_2025_adj,
        df_alloc=df_alloc,
        required_deploy=["GB","BE","NL","DK","DE"],
        tol=1e-6,
    )
)

print("\n=== Missing/extra deployment columns per matrix ===")
print(missing_cols_df if len(missing_cols_df) else "None ✅")

print("\n=== Column-sum issues (should be 1.0) ===")
print(colsum_issues_df if len(colsum_issues_df) else "None ✅")


In [ ]:
# The paper run generates only the supplier-contract count table used in the manuscript.
wb = load_workbook(B1_SUPPLEMENTARY_WORKBOOK, read_only=True, data_only=True)
ws = wb["contracts_summary"]
rows = list(ws.iter_rows(values_only=True))
wb.close()

header = [str(x).strip() for x in rows[3] if x is not None]
data = [list(r[:len(header)]) for r in rows[4:] if r and r[0] is not None]
supplier_contracts_country = pd.DataFrame(data, columns=header)

country_cols = [c for c in ["BE", "DE", "DK", "GB", "NL", "Grand Total"] if c in supplier_contracts_country.columns]
for c in country_cols:
    supplier_contracts_country[c] = pd.to_numeric(supplier_contracts_country[c], errors="coerce").fillna(0).astype(int)

latex = supplier_contracts_country.to_latex(
    index=False,
    escape=True,
    caption="Supplier contracts by component, subcomponent, and deployment country.",
    label="tab:supplier_contracts_country",
    column_format="llrrrrrr",
)

supplier_contracts_path = Path(TAB_ROOT) / "supplier_contracts_country.tex"
supplier_contracts_path.write_text(latex, encoding="utf-8")
print(f"OK supplier contract table saved: {supplier_contracts_path}")

In [ ]:
# B.1.3
plot_gb_local_benchmark(
    flows_raw,
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
)

Sourcing plot


In [ ]:
# 3.1
# Countries shown separately from the EU27 aggregate.
MACRO_SPLIT_COUNTRIES = ("DK", "NL", "DE", "BE", "ES", "FR", "NO")

plot_macro_vertical(
    flows_by_component_hier_all,
    split_iso=MACRO_SPLIT_COUNTRIES,
    filename="3.1_EU_detailed.pdf",
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
)

### MRIO Construction


In [ ]:
# ============================================================================
# MRIO Construction Module 
# ============================================================================
import importlib
import sys
from typing import Literal

for module_name in ['modules.mrio_analysis.industrial',
                    'modules.mrio_analysis.charts',
                    'modules.mrio_analysis.employment', 
                    'modules.mrio_analysis.mrio']:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

from modules.mrio_analysis.industrial import (
    run_industrial,
    results_by_origin_industry_groups,
    plot_eu27_composition_groups,
)

from modules.config import HELPERS_DIR

from modules.mrio_analysis.mrio import (
    build_or_load_mrio,
    extract_E,
    calculate_A,
    calculate_C,
    calculate_vn_pj,
    calculate_L,
    _row_from_v,
    load_classifications,
    plot_lb_box_all_industries,
)

from modules.mrio_analysis.employment import (
    show_preferred_dataflows,
    load_emp_eurostat,
    fetch_cl_area_iso3,
    reduce_ilo_core_using_eurostat,
    build_ISO3_LIST,
    pull_flows_deductive,
    process_ilo_long,
    map_nace_to_cpa_inplace,
    fit_emp_eu_to_mrio,
    build_wages,
    plot_wages_dispersion_boxplot_sorted,
    make_db_year_source_countrylist,
    load_emp_eurostat_lfsa_egan22d,
)

print("✓ All mrio_analysis modules imported and data loaded successfully")

In [ ]:
# ==============================================================
# RUN Industrial allocation
# ==============================================================
ind_ind = True

ind_codes, final_codes, primary_inputs, ind_labels = load_classifications(HELPERS_DIR, use_industry_matrix=ind_ind)

final_cost_df_agg, component_groups, subcomponents_dict = run_industrial(
    ROOT=str(HELPERS_DIR),
    flows_by_component_hier_all=flows_by_component_hier_all,
    df_alloc=df_alloc,
    ind_ind=ind_ind,  
)
tot_dep = final_cost_df_agg.groupby("deployment_country", as_index=False)["value_alloc"].sum()
tot_dep.sort_values("value_alloc", ascending=False)

In [ ]:
# C.1.1
plot_eu27_composition_groups(
    "BE",
    final_cost_df_agg,
    SAVE_FIGURES,
    CLOSE_FIGURES,
    fig_name="C.1.1",
    ind_ind=ind_ind,
)

Classification results


In [ ]:
# ==============================================================
# REGION INTO ANALYSIS
# ==============================================================

EU27 = {"AT", "BE", "BG", "CY", "CZ", "DE", "DK", "EE", "ES", "FI", 
        "FR", "GR", "HR", "HU", "IE", "IT", "LT", "LU", "LV", "MT", 
        "NL", "PL", "PT", "RO", "SE", "SI", "SK"} 

EU_EFTA_UK = EU27 | {"GB", "NO", "CH", "IS", "LI"}

# Offshore Financial Centers
# https://www.imf.org/external/pubs/ft/wp/2007/wp0787.pdf
# https://www.ecb.europa.eu/press/economic-bulletin/articles/2020/html/ecb.ebart202002_01~1a58c02776.en.html
EU_OFC = {"CH", "LU", "MT", "IE"}
include_eu_ofc = False  # Set to False to exclude EU OFCs from the Europe group
EUROPE_GROUP = EU_EFTA_UK | {"RS", "EU-OFC"}
CORE = { 
    # EU27
    "AT", "BE", "BG", "CY", "CZ", "DE", "DK", "EE", "ES", "FI", "FR", "GR",
    "HR", "HU", "IT", "LT", "LV", "NL", "PL", "PT", "RO",
    "SE", "SI", "SK",
    # EU27 financial hubs
    "LU", "MT", "CH", "IE",
    # Relevant for Europe
    "GB", "NO", "TR", "RU", "IS", "RS",
    # Relevant trade partners
    "CN", "US", "JP", "CA", "KR", "IN", "AU","ZA","ID",
    ## Less relevant trade partners ##
    # # Balkans
    # "AL", "ME", "MK", 
    # # ROW
    "SA",  
    # # South America
    "AR", "BR", "MX",
    # Value added components
    "W2"
}

LOW_INCOME = {
    "TR", "BR", "CN",
    "MX", "ZA", "RU", "SA",
    "ID", "IN",
}

In [ ]:
# Source: https://circabc.europa.eu/ui/group/cec66924-a924-4f91-a0ef-600a0531e3ba/library/ff0e2b4f-b572-424a-8f09-4b484ceabc7e/details?download=true

MRIO = build_or_load_mrio(
    CORE=CORE,
    include_eu_ofc=include_eu_ofc,
    EU_OFC=EU_OFC,
    use_cache=True,
    refresh_cache=FIRST_MRIO,
    save_raw_parquet=FIRST_MRIO,
    validate=True,
    use_industry_matrix=ind_ind,
    download_matrix=FIRST_MRIO,
)

In [ ]:
Scope = Literal["global", "local"]

EUROSTAT


In [ ]:
emp_eu = load_emp_eurostat(CORE,EU_OFC, min_nace=60)

ILOSTAT


In [ ]:
PREFERRED_DATAFLOWS = [
    "DF_EES_TEES_SEX_EC2_NB",
    "DF_EMP_TEMP_SEX_ECO_NB",       # ISIC level 2 (employees)
    "DF_EMP_2EMP_SEX_ECO_NB",
    # "DF_EMP_TEMP_SEX_EC2_NB", 
    # "DF_EMP_TEMP_SEX_STE_EC2_NB",
    # "DF_EMP_TEMP_SEX_IFL_EC2_NB",
    # "DF_EMP_TEMP_SEX_EST_EC2_NB",
    # "DF_EMP_TEMP_ECO_OCU_NB",
    # "DF_EMP_TEMP_SEX_AGE_ECO_NB",
    # "DF_EMP_TEMP_SEX_STE_ECO_NB",# total employment, ECO
]

show_preferred_dataflows(PREFERRED_DATAFLOWS,refresh_cache=RECOMPUTE,)

In [ ]:
# =========================
# RUN EMPLOYMENT
# =========================
areas_iso3 = fetch_cl_area_iso3()
res = reduce_ilo_core_using_eurostat(emp_eu, CORE, min_nace=60)
ISO3_LIST = [c3 for c3 in res["ISO3_LIST_REDUCED"] if (c3 in areas_iso3)] 

YEAR=2023

ISO3_LIST, missing_iso2 = build_ISO3_LIST(emp_eu, CORE, min_nace=50)
ILO_LONG, ILO_LOG, ISO3_LEFT = pull_flows_deductive(PREFERRED_DATAFLOWS, ISO3_LIST, year=YEAR, freq="A")
emp_ilo_raw  = process_ilo_long(ILO_LONG, drop_x=False, source_db="ILOSTAT_SDMX")

In [ ]:
# =========================
# CALCULATE EMPLOYMENT
# =========================
E_MRIO = extract_E(MRIO)
mrio_prd = pd.Index(E_MRIO.index.get_level_values("prd_use")).astype(str).str.strip().unique()
mrio_prd_set = set(mrio_prd)
emp_ilo = map_nace_to_cpa_inplace(emp_ilo_raw, mrio_prd_set, ind_ind=ind_ind)

emp_eu_clean = fit_emp_eu_to_mrio(emp_eu, mrio_prd_set, ind_ind=ind_ind)
emp_ilo_clean = fit_emp_eu_to_mrio(emp_ilo, mrio_prd_set, ind_ind=ind_ind)

emp_core = pd.concat([emp_eu_clean, emp_ilo_clean], ignore_index=True)
 
wages, report, empty = build_wages(
    emp_core,
    E_MRIO,
    mrio_prd_set,
    EU_OFC,
    ind_ind=ind_ind,
    use_cache=False,
    recompute=RECOMPUTE,
    cache_key=f"wages_{YEAR}_{'ind' if ind_ind else 'prd'}",
)

In [ ]:
plot_wages_dispersion_boxplot_sorted(wages,eu_countries=EUROPE_GROUP, stat="median", 
                                     filename="C.2.1.pdf",
                                     save_figures=SAVE_FIGURES,
                                     close_figures=CLOSE_FIGURES)

In [ ]:
w = wages.reset_index()
print("industries:", w["prd_use"].nunique())
print("countries:", w["destination_region"].nunique())

Model computations


In [ ]:
# ==============================================================
# Macro
A_MRIO  = calculate_A(MRIO)
C_MRIO  = calculate_C(MRIO)
V_MRIO  = calculate_vn_pj(MRIO)
L1_MRIO = calculate_L(A_MRIO, V_MRIO, C_MRIO, system_type=1)  # Type I

# ==============================================================
# Factors of production 
# Capital income coefficient (may be negative in some industries)
K_MRIO  = _row_from_v(V_MRIO, "B2A3G").reindex(L1_MRIO.columns).fillna(0.0)
# Labour income coefficient (typically non-negative)
LB_MRIO = _row_from_v(V_MRIO, "D1").reindex(L1_MRIO.columns).fillna(0.0)
# Taxes 
T_MRIO = _row_from_v(V_MRIO, "D29X39").reindex(L1_MRIO.columns).fillna(0.0)
# Factor-income capture coefficient (labour + capital) (impose non-negative)
I_MRIO = LB_MRIO + K_MRIO
# Gross value added (GVA)
VA_MRIO = LB_MRIO + K_MRIO + T_MRIO

# Wages: align explicit country and ROW wages to the MRIO industry columns.
W_MRIO = wages.reindex(L1_MRIO.columns)

dest_level = "destination_region" if "destination_region" in L1_MRIO.columns.names else 0
dest_regions = L1_MRIO.columns.get_level_values(dest_level).astype(str).str.strip()
row_cols = (dest_regions == "ROW")
covered = W_MRIO.notna()

print(
    "Wage coverage in MRIO columns:",
    f"{int(covered.sum())}/{len(W_MRIO)} total;",
    f"ROW {int((covered & row_cols).sum())}/{int(row_cols.sum())}"
)

if row_cols.any() and not W_MRIO.loc[row_cols].notna().any():
    raise ValueError("MRIO has ROW columns, but wages has no aligned ROW wages. Rebuild wages before computing GLOBAL jobs.")

W_MRIO = W_MRIO.fillna(0.0)
W_MRIO_CACHE_KEY = f"wages_{int(pd.util.hash_pandas_object(W_MRIO, index=True).sum()) & 0xFFFFFFFF:08x}"
print("Wage cache key:", W_MRIO_CACHE_KEY)

In [ ]:
plot_lb_box_all_industries(
    W_MRIO,
    industries=wages['BE'].index,
    europe_group=EUROPE_GROUP,
    filename="C.2.2.pdf",
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
    ylabel="Compensations (kEUR/year)"
)

In [ ]:
mrio_origin_set = set(MRIO.index.get_level_values("origin_region").astype(str).unique())
final_cost_df_agg["origin_country"] = (
    final_cost_df_agg["origin_country"].astype(str).str.strip()
      .replace({"UK": "GB", "RoW": "ROW"})
      .where(lambda s: s.isin(mrio_origin_set), "ROW")
)

### BASELINE RESULTS

In [ ]:
# ============================================================================
# Impact Analysis Module Imports
# ============================================================================

import importlib
import sys

for module_name in [
    "modules.mrio_analysis.mrio",
    "modules.results.results",
    "modules.results.scenarios",
    "modules.cost_model.charts",
    "modules.mrio_analysis.maps",
    "modules.latex_style",
]:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])

from modules.mrio_analysis.mrio import (
    build_y_shock_levels,
    compute_all_results,
)

from modules.results.results import (
    build_all_indicators_table_effect_and_comp_columns_from_by_origin,
    compute_industry_effects_by_industry,
    plot_industry_multipliers_twin,
    plot_and_save_eu27_capture_share_figures,
)

import modules.mrio_analysis.maps as maps
from modules.mrio_analysis.maps import configure_paper_map_figures

from modules.latex_style import (
    df_to_latex,
)

print("✓ Impact Analysis modules imported successfully")

Industrial multipliers


In [ ]:
# ============================================================================
# ============= Code industries selected for the paper analysis ============== 
# ============================================================================
ind_rev = ['B','C24','C25','C27','C28','C33','D35','F',
           'H49','H50','H52','K65','M69_70','M71','M72']

In [ ]:
df_by_industry = compute_industry_effects_by_industry(
    ind_rev,
    mrio=MRIO,
    L1_MRIO=L1_MRIO,
    VA_MRIO=VA_MRIO,
    LB_MRIO=LB_MRIO,
    W_MRIO=W_MRIO,
    indicator="Output",
    shock_eur=1_000_000.0,
    include_deployments=EU27,
    subtract_self=False,
)

Plot

In [ ]:
# C.1.2
fig, ax = plot_industry_multipliers_twin(
    df_by_industry,
    industries=ind_rev,
    ind_labels=ind_labels,
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
)


Demand function


In [ ]:
y_shock_lvl, y_shock_sub_lvl = build_y_shock_levels(final_cost_df_agg, CORE_DEPLOYMENT, MRIO)

Baseline calculation

In [ ]:
results_lvl, results_sub_lvl, results_by_origin_lvl, results_by_origin_sub_lvl = compute_all_results(
    y_shock_lvl, y_shock_sub_lvl, YEAR, L1_MRIO, VA_MRIO, LB_MRIO, W_MRIO,
)

Table

In [ ]:
REGION_ORDER = ["EU27", "UK", "RoE", "RoW", "Global", "EU27%", "Europe %", "RoW %"]
COMPONENT_ORDER = ["Total", "Devex", "Turbine", "BoP", "Installation", "O&M"]
INDICATORS = ["GVA", "Jobs"]
lvl = 2
ANNUALIZE = False

NOTE = (
    r"\textit{Note:} GVA is reported in EUR million and employment in "
    r"thousand job-years supported. RoE denotes the rest of Europe, "
    r"excluding the EU27 and the United Kingdom; RoW denotes the rest of the world."
)

# =========================
# EFFECT + COMPONENT COLUMNS
# Direct / Indirect + Total by component
# =========================
effect_comp_cols = build_all_indicators_table_effect_and_comp_columns_from_by_origin(
    results_by_origin_sub_lvl,
    core=CORE_DEPLOYMENT,
    connected_country=connected_country,
    indicators=INDICATORS,
    metric_order=REGION_ORDER,
    region_order=REGION_ORDER,
    annualize=ANNUALIZE,
    lifetime_years=lifetime_years,
)

summary_path = Path(TAB_ROOT) / "summary_indicators_compcols_total.tex"

latex = df_to_latex(
    effect_comp_cols,
    caption="Cumulative 25-year economic impacts of a 1 GW European OWF by effect type and component.",
    label="tab:summary_indicators_compcols_total",
    note=NOTE,
)

summary_path.write_text(latex, encoding="utf-8")

print("OK combined effect/component table saved:")
print(summary_path)

Plots by component and subcomponent

In [ ]:
capture_share_figures = plot_and_save_eu27_capture_share_figures(
    results_by_origin_sub_lvl=results_by_origin_sub_lvl,
    connected_country=connected_country,
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
)

Geographical distribution


In [ ]:
SAVE_MAPS = False

In [ ]:
components_map = {
    "Turbine": ["Turbine"],
    "Devex & O&M": ["Devex", "O&M"],
    "Balance of plant": ["BoP"],
    "Installation": ["Installation"],
}

EU maps

In [ ]:
# D.1 EU27 maps
maps.plot_paper_eu27_component_share_maps(
    results_by_origin=results_by_origin_lvl,
    components_map=components_map,
    core_deployment=CORE_DEPLOYMENT,
    eu27=EU27,
    connected_country=connected_country,
    lifetime_years=lifetime_years,
    save=SAVE_MAPS,
    close=CLOSE_FIGURES,
)

Global maps

In [ ]:
# D.2 global maps
maps.plot_paper_world_component_share_maps(
    results_by_origin=results_by_origin_lvl,
    components_map=components_map,
    core_deployment=CORE_DEPLOYMENT,
    eu27=EU27,
    connected_country=connected_country,
    save=SAVE_MAPS,
    close=CLOSE_FIGURES,
)

### REALLOCATION SCENARIOS


In [ ]:
import importlib
import sys

import modules.results.scenarios as scenarios

scenarios = importlib.reload(scenarios)

from modules.results.scenarios import (
    maybe_run_or_load,
    maybe_run_eu27_side_from_net,
    build_baseline_panels,
    plot_and_save_d3_offshoring_figures,
    plot_and_save_eu27_component_delta_area_figures,
    plot_and_save_eu27_component_multiplier_figures,
    plot_and_save_eu27_turbine_subcomponent_multiplier_figures,
    build_eu27_resilience_accounting_table,
    LEVELS,
)

print("OK Scenarios Analysis modules imported successfully")
print(f"  - LEVELS: {LEVELS}")

In [ ]:
# =============================
# Scenario Inputs
# =============================

scope = "EU27"
exclude_deployments_resilience = ()

SCENARIO_RECOMPUTE = RECOMPUTE
EU_SIDE_RECOMPUTE = RECOMPUTE

DELTA_BASE = 0      # Baseline sourcing case used as comparison point.
DELTA_RESIL = 100   # Full resilience/reshoring scenario.
DELTA_OUT = -100    # Full offshoring scenario.
DELTAS = sorted(set(range(DELTA_OUT, DELTA_RESIL + 1, 5)))

MARKUP_EU = 0.10

# Scenario aggregation levels used by the paper figures.
# lvl_comp=2: broad components used in D.3, 3.4, and the resilience table.
# lvl_sub=1: intermediate subcomponent groups used in 3.5.
# lvl_sim=0: most detailed capacity level used inside the scenario simulation.
# Changing these levels changes the aggregation of the scenario outputs and figures.
lvl_comp = 2
lvl_sub = 1
lvl_sim = 0

# RoW countries shown separately in the D.3 offshoring figures.
# Labels are resolved inside the scenarios module from CODE_TO_NAME.
TARGETS_SELECTED = ["CN", "KR", "IN", "ZA"]

Scenario simulation

In [ ]:
# =============================
# Run/load EU27 net sourcing scenarios
# =============================
sd_base = {}
sr_base = {}

sd_base[scope], sr_base[scope] = maybe_run_or_load(
    cache_dir="EU27_resilience",
    cache_base_dir=RESULTS_ROOT,
    force_recompute=SCENARIO_RECOMPUTE,
    EUROPE=EU27,
    EU27=EU27,
    df_alloc=df_alloc,
    flows_by_component_hier_all=flows_by_component_hier_all,
    ROOT=str(HELPERS_DIR),
    deltas_pp=DELTAS,
    L1_MRIO=L1_MRIO,
    VA_MRIO=VA_MRIO,
    LB_MRIO=LB_MRIO,
    W_MRIO=W_MRIO,
    receiver_policy="baseline_mix",
    results_by_origin_lvl=results_by_origin_lvl,
    results_by_origin_sub_lvl=results_by_origin_sub_lvl,
    capacity_lvl=lvl_sim,
    connected_country=connected_country,
    eu_spend_reduction_pct=MARKUP_EU,
    preserve_total=False,
    mrio=MRIO,
    core=CORE_DEPLOYMENT,
    annualize=ANNUALIZE,
    lifetime_years=lifetime_years,
)


Gross / net effects

In [ ]:
# =============================
# EU27-side reallocation effects
# This replaces sr_gross for the delta-area decomposition figures.
# It isolates the EU27 side of the signed reallocation:
#   delta > 0: EU27 expansion from reshoring
#   delta < 0: EU27 contraction from offshoring
# =============================
sd_eu_side = {}
sr_eu_side = {}

sd_eu_side[scope], sr_eu_side[scope] = maybe_run_eu27_side_from_net(
    net_scenarios_delta=sd_base[scope],
    eu_countries=EU27,
    cache_dir=f"{scope}_eu27_side",
    cache_base_dir=RESULTS_ROOT,
    force_recompute=EU_SIDE_RECOMPUTE,
    deltas_pp=DELTAS,
    L1_MRIO=L1_MRIO,
    VA_MRIO=VA_MRIO,
    LB_MRIO=LB_MRIO,
    W_MRIO=W_MRIO,
    mrio=MRIO,
    core=CORE_DEPLOYMENT,
    annualize=ANNUALIZE,
    lifetime_years=lifetime_years,
)

Plot Global effects by project segments - all scenarios

In [ ]:
# =============================
# Build D.3 panels
# D.3 separates selected RoW countries and EU27 indirect effects.
# =============================
_, PANEL_COMP = build_baseline_panels(
    sr_base[scope],
    split_row=TARGETS_SELECTED,
    EU27=EU27,
    connected_country=connected_country,
    exclude_deployments=exclude_deployments_resilience,
    add_eu_indirect_split=True,
    lvl_plot=lvl_comp,
)

PANEL_COMP = PANEL_COMP.copy()
PANEL_COMP.loc[PANEL_COMP["Component"].isin(["Devex", "O&M"]), "Component"] = "Devex+O&M"

ABS_COLS = [c for c in PANEL_COMP.columns if c.startswith("abs_")]
TOTAL_KEYS = ["delta_pp", "scenario", "Region"]

PANEL_COMP_TOTAL = PANEL_COMP.groupby(TOTAL_KEYS, as_index=False)[ABS_COLS].sum()
PANEL_COMP_TOTAL["Component"] = "Total"

PANEL_COMP_D3 = pd.concat([PANEL_COMP, PANEL_COMP_TOTAL], ignore_index=True, sort=False)

In [ ]:
# D.3 figures: offshoring effects
plot_and_save_d3_offshoring_figures(
    PANEL_SUB=PANEL_COMP_D3,
    split_row=TARGETS_SELECTED,
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
)

print("OK D.3 figures generated.")

Plot EU27 effects by project segments - all scenarios

In [ ]:
# =============================
# Component-level panels
# One raw version is built for net effects and one for EU27-side effects.
# Copies are then used for figures and tables to avoid recomputing panels.
# =============================
_, PANEL_COMP_NET_RAW = build_baseline_panels(
    sr_base[scope],
    EU27=EU27,
    connected_country=connected_country,
    exclude_deployments=exclude_deployments_resilience,
    lvl_plot=lvl_comp,
)

PANEL_COMP_NET_RAW = PANEL_COMP_NET_RAW[
    PANEL_COMP_NET_RAW["scenario"] == "baseline"
].copy()

_, PANEL_COMP_EU_SIDE_RAW = build_baseline_panels(
    sr_eu_side[scope],
    EU27=EU27,
    connected_country=connected_country,
    exclude_deployments=exclude_deployments_resilience,
    lvl_plot=lvl_comp,
)

PANEL_COMP_EU_SIDE_RAW = PANEL_COMP_EU_SIDE_RAW[
    PANEL_COMP_EU_SIDE_RAW["scenario"] == "baseline"
].copy()

In [ ]:
# =============================
# Component panels for 3.4 figures
# Devex and O&M are grouped only for these figures.
# =============================
PANEL_COMP_NET = PANEL_COMP_NET_RAW.copy()
PANEL_COMP_NET.loc[PANEL_COMP_NET["Component"].isin(["Devex", "O&M"]), "Component"] = "Devex+O&M"

PANEL_COMP_EU_SIDE = PANEL_COMP_EU_SIDE_RAW.copy()
PANEL_COMP_EU_SIDE.loc[PANEL_COMP_EU_SIDE["Component"].isin(["Devex", "O&M"]), "Component"] = "Devex+O&M"

plot_and_save_eu27_component_delta_area_figures(
    PANEL_NET=PANEL_COMP_NET,
    PANEL_EU_SIDE=PANEL_COMP_EU_SIDE,
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
)

In [ ]:
# 3.4 figures: EU27 gross/net effects by component
plot_and_save_eu27_component_delta_area_figures(
    PANEL_NET=PANEL_COMP_NET,
    PANEL_EU_SIDE=PANEL_COMP_EU_SIDE,
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
)

print("OK 3.4 EU27 gross/net absolute-effect component figures generated.")

Plot EU27 effect by components

In [ ]:
# =============================
# Subcomponent panels: L1 net and EU27-side effects
# Used for 3.5 component/subcomponent decomposition figures.
# =============================
PANEL_SUB_L1_NET, _ = build_baseline_panels(
    sr_base[scope],
    EU27=EU27,
    connected_country=connected_country,
    exclude_deployments=exclude_deployments_resilience,
    lvl_plot=lvl_sub,
)

PANEL_SUB_L1_NET = PANEL_SUB_L1_NET[
    PANEL_SUB_L1_NET["scenario"] == "baseline"
].copy()

PANEL_SUB_L1_EU_SIDE, _ = build_baseline_panels(
    sr_eu_side[scope],
    EU27=EU27,
    connected_country=connected_country,
    exclude_deployments=exclude_deployments_resilience,
    lvl_plot=lvl_sub,
)

PANEL_SUB_L1_EU_SIDE = PANEL_SUB_L1_EU_SIDE[
    PANEL_SUB_L1_EU_SIDE["scenario"] == "baseline"
].copy()

print("L1 net components:", sorted(PANEL_SUB_L1_NET["Component"].unique()))
print("L1 EU-side components:", sorted(PANEL_SUB_L1_EU_SIDE["Component"].unique()))

In [ ]:
# 3.5 figures: EU27 gross/net effects by component group
plot_and_save_eu27_component_multiplier_figures(
    PANEL_NET=PANEL_SUB_L1_NET,
    PANEL_GROSS=PANEL_SUB_L1_EU_SIDE,
    delta_resil=DELTA_RESIL,
    delta_out=DELTA_OUT,
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
)

print("OK 3.5 component resilience gross/net figures generated.")

Plot EU27 effect by subcomponents

In [ ]:
# =============================
# Subcomponent panels: L0 net and EU27-side effects
# Used for 3.6 turbine detailed subcomponent figures.
# =============================
PANEL_SUB_L0_NET, _ = build_baseline_panels(
    sr_base[scope],
    EU27=EU27,
    connected_country=connected_country,
    exclude_deployments=exclude_deployments_resilience,
    lvl_plot=lvl_sim,
)

PANEL_SUB_L0_EU_SIDE, _ = build_baseline_panels(
    sr_eu_side[scope],
    EU27=EU27,
    connected_country=connected_country,
    exclude_deployments=exclude_deployments_resilience,
    lvl_plot=lvl_sim,
)

In [ ]:
# 3.6 figures: EU27 gross/net effects by turbine subcomponent
plot_and_save_eu27_turbine_subcomponent_multiplier_figures(
    PANEL_NET=PANEL_SUB_L0_NET,
    PANEL_GROSS=PANEL_SUB_L0_EU_SIDE,
    delta_resil=DELTA_RESIL,
    delta_out=DELTA_OUT,
    save=SAVE_FIGURES,
    close=CLOSE_FIGURES,
)

print("OK 3.6 turbine subcomponent gross/net figures generated.")

Accounting table

In [ ]:
# =============================
# Resilience accounting table
# =============================
summary = build_eu27_resilience_accounting_table(
    PANEL_NET=PANEL_COMP_NET_RAW,
    PANEL_EU_SIDE=PANEL_COMP_EU_SIDE_RAW,
    delta_base=DELTA_BASE,
    delta_resil=DELTA_RESIL,
)

latex = df_to_latex(
    summary,
    caption="EU27 effects of the full resilience sourcing scenario for a 1 GW European OWF.",
    label="tab:eu27_resilience_accounting",
    note=(
        r"\textit{Note:} GVA is reported in EUR million and employment in "
        r"thousand job-years supported. The net effect equals the gross effect "
        r"after upstream trade effects."
    ),
)
summary_path = Path(TAB_ROOT) / "summary_eu27_resilience_accounting.tex"
summary_path.write_text(latex, encoding="utf-8")
print(f"OK EU27 resilience effects table saved: {summary_path}")